# 📦 Notebook 1: Dataset Preparation & Augmentation
**Cattle Vision Pipeline — Texas A&M University**  
**Author:** Sreenivasulu Reddy Settipalli | sreenivasulureddy@tamu.edu

This notebook covers:
1. Downloading the cattle dataset from Roboflow
2. Exploring & visualizing annotations
3. Applying Albumentations augmentation pipeline
4. Preparing the final train/val/test split for YOLO training

In [ ]:
# Install dependencies
!pip install roboflow ultralytics albumentations supervision -q

In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
import albumentations as A
from pathlib import Path
from roboflow import Roboflow
import supervision as sv

print('Libraries loaded ✅')

## 1. Download Dataset from Roboflow

In [ ]:
# Initialize Roboflow — replace with your API key
rf = Roboflow(api_key='YOUR_ROBOFLOW_API_KEY')

# Download cattle detection dataset (YOLOv11 format)
project = rf.workspace('cattle-detection').project('cattle-yolov11')
dataset = project.version(3).download('yolov11')

DATA_DIR = Path(dataset.location)
print(f'Dataset saved to: {DATA_DIR}')
print(f'Train: {len(list((DATA_DIR/"train/images").glob("*.jpg")))} images')
print(f'Val:   {len(list((DATA_DIR/"valid/images").glob("*.jpg")))} images')

## 2. Visualize Sample Annotations

In [ ]:
CLASS_NAMES = ['cattle', 'calf', 'bull']
CLASS_COLORS = sv.ColorPalette.DEFAULT

def visualize_sample(image_path, label_path):
    image = cv2.imread(str(image_path))
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    h, w = image.shape[:2]

    boxes, class_ids = [], []
    with open(label_path) as f:
        for line in f:
            cls, cx, cy, bw, bh = map(float, line.strip().split())
            x1 = int((cx - bw/2) * w);  y1 = int((cy - bh/2) * h)
            x2 = int((cx + bw/2) * w);  y2 = int((cy + bh/2) * h)
            boxes.append([x1, y1, x2, y2])
            class_ids.append(int(cls))

    detections = sv.Detections(
        xyxy=np.array(boxes),
        class_id=np.array(class_ids)
    )
    annotator = sv.BoxAnnotator()
    labels = [CLASS_NAMES[c] for c in class_ids]
    annotated = annotator.annotate(image.copy(), detections, labels=labels)
    return annotated

# Show 4 random samples
img_paths = list((DATA_DIR / 'train/images').glob('*.jpg'))[:4]
fig, axes = plt.subplots(1, 4, figsize=(20, 5))
for ax, img_p in zip(axes, img_paths):
    lbl_p = DATA_DIR / 'train/labels' / (img_p.stem + '.txt')
    annotated = visualize_sample(img_p, lbl_p)
    ax.imshow(annotated);  ax.axis('off')
plt.suptitle('Sample Training Annotations', fontsize=14)
plt.tight_layout()
plt.show()

## 3. Augmentation Pipeline (Albumentations)

In [ ]:
# Define augmentation pipeline for livestock imagery
train_transform = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.RandomBrightnessContrast(brightness_limit=0.3, contrast_limit=0.3, p=0.6),
    A.HueSaturationValue(hue_shift_limit=10, sat_shift_limit=30, val_shift_limit=20, p=0.5),
    A.GaussNoise(var_limit=(10, 50), p=0.3),
    A.MotionBlur(blur_limit=5, p=0.2),      # simulate camera motion
    A.RandomRain(p=0.1),                     # weather robustness
    A.RandomSunFlare(p=0.05),               # outdoor lighting
    A.CLAHE(clip_limit=2, p=0.3),
    A.Resize(640, 640),
], bbox_params=A.BboxParams(format='yolo', label_fields=['class_labels']))

# Apply to 3 sample images and visualize
img_paths = list((DATA_DIR / 'train/images').glob('*.jpg'))[:3]
fig, axes = plt.subplots(3, 2, figsize=(12, 15))

for row, img_p in enumerate(img_paths):
    original = cv2.cvtColor(cv2.imread(str(img_p)), cv2.COLOR_BGR2RGB)
    axes[row][0].imshow(original); axes[row][0].set_title('Original'); axes[row][0].axis('off')

    augmented = train_transform(image=original, bboxes=[], class_labels=[])['image']
    axes[row][1].imshow(augmented); axes[row][1].set_title('Augmented'); axes[row][1].axis('off')

plt.suptitle('Augmentation Pipeline Preview', fontsize=14)
plt.tight_layout()
plt.show()
print('Augmentation pipeline ready ✅')

## 4. Class Distribution Analysis

In [ ]:
from collections import Counter

def count_classes(label_dir):
    counts = Counter()
    for lbl in Path(label_dir).glob('*.txt'):
        with open(lbl) as f:
            for line in f:
                cls = int(line.split()[0])
                counts[CLASS_NAMES[cls]] += 1
    return counts

train_counts = count_classes(DATA_DIR / 'train/labels')
val_counts   = count_classes(DATA_DIR / 'valid/labels')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, (counts, title) in zip(axes, [(train_counts, 'Train'), (val_counts, 'Val')]):
    ax.bar(counts.keys(), counts.values(), color=['#4CAF50','#FF9800','#9C27B0'])
    ax.set_title(f'{title} Class Distribution')
    ax.set_ylabel('Count')
    for k, v in counts.items():
        ax.text(list(counts.keys()).index(k), v + 5, str(v), ha='center', fontweight='bold')
plt.tight_layout()
plt.show()
print('Train:', dict(train_counts))
print('Val:  ', dict(val_counts))

## ✅ Summary

Dataset is ready for YOLO training:
- Downloaded from Roboflow in YOLOv11 format
- Visualized annotations with supervision
- Augmentation pipeline configured for outdoor livestock imagery
- Class distribution verified

**Next:** Run `02_yolo_training.ipynb` to fine-tune YOLOv11 on this dataset.